## **Natural Language-Derived Entity Placement in Vector-Space**

### Project Overview

The tokenization-vectorization scheme in Natural Language Processing (NLP) is incredible: organizing language in multi-dimensional vector-space by semantic/pragmatic meaning. This project seeks to orient whole entities by the same paradigm, pooling natural language descriptions into a singular vector that serves as a **qualified** representation of a given entity.

### The Hypothetical

Imagine, if you will, that we manage a home goods store, which sells a number of sofas. Each sofa has a collection of reviews, left by the people who have purchased that particular product from us. We will use these reviews to create a representation of these products in vector-space.

For this scenario, we might design the following NLP-inspired workflow:
* break each review into tokens
* obtain a vectorized representation for each of these tokens
* pool these representations to arrive at an aggregate representation for each review
* pool these review-level representations to arrive at an aggregate representation for the overall product

### Choices, Choices

Before applying the workflow above, we must choose a tokenizer/model and pooling method to use. These choices have significance: for example, different tokenizers might break the same review into different token sets, and different models might represent the same token differently in vector-space. Pooling methods can also function differently. All of these considerations can affect the ultimate product representation.

To inform these choices, I have created a synthetic textset of nine 'reviews'. After applying different combinations of model/pooling to tokenize, vectorize, and pool them at the review-level, we can compare the vector representations for each of these nine reviews to one another (via cosine similarity) in order to see how their orientations differ, and then select a particular model/pooling combination to apply to real-world data.

The following process is automated in the below code cell: for each model/pooling combination, the reviews are tokenized, vectorized, and pooled on the review-level, with one vector representation for each of the nine reviews. The cosine similarities for unique pairings of these reviews are calculated and displayed in the DataFrame. Note that reviews are not compared to themselves (ex. the first review would not be compared for cosine similarity with itself). More reviews, tokenizers/models, and pooling methods, may be added below - see the TODOs in the code cell.

**The code for mean_pool was taken/adapted from the model card for the ['all-mpnet-base-v2'](https://huggingface.co/sentence-transformers/all-mpnet-base-v2) transfomer model.*<br>
***Models in model_li were found on Hugging Face: [sentence-transformers/all-mpnet-base-v2](https://huggingface.co/sentence-transformers/all-mpnet-base-v2) | [sentence-transformers/nli-mpnet-base-v2](https://huggingface.co/sentence-transformers/nli-mpnet-base-v2) | [princeton-nlp/sup-simcse-bert-base-uncased](https://huggingface.co/princeton-nlp/sup-simcse-bert-base-uncased)*

In [1]:
## A synthetic textset of nine reviews associated with a fictional sofa.

# TODO: more reviews may be added here if desired
reviews = ["comfortable", \
           "This is the most comfortable sofa ever.", \
           "This is the most uncomfortable sofa ever.", \
           "comfortable", \
           "very comfortable", \
           "Loved this sofa, definitely a good one.", \
           "Hated this sofa, comfortable but should not buy because the color is obscene.", \
           "Cozy and nice.", \
           "Cozy but would not buy."]

## Imports that we will use.
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
import pandas as pd

## A list of models across which to compare performance.

# TODO: more models may be added if desired
model_li = ['sentence-transformers/all-mpnet-base-v2', \
            'sentence-transformers/nli-mpnet-base-v2', \
            'princeton-nlp/sup-simcse-bert-base-uncased']

## A couple of pooling functions across which to compare performance.

# TODO: more pooling methods may be added if desired, also review next TODO if adding

# mean pooling (token embeddings -> review-level embeddings)
def mean_pool(token_embeddings, attention_mask):
    # adds a view across hidden_size to attention_mask
    attention_mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    # multiplies tokens by attention_mask and averages across tokens with attention '1'
    return torch.sum(token_embeddings * attention_mask, 1) / torch.clamp(attention_mask.sum(1), min=1e-9)

# CLS pooling (token embeddings -> review-level embeddings)
def cls_pool(token_embeddings, attention_mask):
    # zeroes out attention for all tokens but the first
    for i in range(attention_mask.shape[0]):
        for j in range(1, attention_mask.shape[1]):
            attention_mask[i][j] = 0
    # calls mean_pool, averaging across a single token (per attention_mask)
    return mean_pool(token_embeddings, attention_mask)

## A list of pooling methods across which to compare performance.

# TODO: pooling methods added above must also be added to the below list
pool_li = [mean_pool, cls_pool]

## Construct a DataFrame 'compare' to hold cosine similarities across unique pairings of reviews, to compare performance across combinations of model/pooling.

sentence_1 = []
sentence_2 = []
seen = []
for i in range(len(reviews)):
    for j in range(i + 1, len(reviews)):
        # if tuple of reviews[i] and reviews[j] in ANY order is NOT in 'seen'
        if(not( ((reviews[i], reviews[j]) in seen) or ((reviews[i], reviews[j]) in seen) )):
            # append the reviews and where they appear in reviews to sentence_1 and sentence_2
            sentence_1.append((reviews[i], i))
            sentence_2.append((reviews[j], j))
            # add this pairing to seen
            seen.append((reviews[i], reviews[j]))
compare = pd.DataFrame({'sentence_1': sentence_1, 'sentence_2': sentence_2})

## Now, we automate the tokenizing, vectorizing, and pooling using different combinations of model/pooling and generating cosine similarity across 
## unique review pairings.
## We can analyze these cosine similarities to pick a model/pooling combination to use on real-world data.

for i in range(len(model_li)):
    # choosing a tokenizer/model
    tokenizer = AutoTokenizer.from_pretrained(model_li[i])
    model = AutoModel.from_pretrained(model_li[i])

    # tokenizing the reviews
    tok_reviews = tokenizer(reviews, padding=True, truncation=True, return_tensors='pt')

    # using the model to get embeddings for the tokens
    with torch.no_grad():
        tok_embed_reviews = model(**tok_reviews)

    # choosing a pooling function
    for j in range(len(pool_li)):
        pool_func = pool_li[j]

        # apply pooling function (token embeddings -> review-level embeddings) and take the L2 norm
        rev_embed_reviews = F.normalize(pool_func(tok_embed_reviews.last_hidden_state, tok_reviews.attention_mask), p=2, dim=1)

        cos_sim = []
        for k in range(compare.shape[0]):
            # the indices of each sentence in a particular row of 'compare'
            index_1 = compare.iloc[k, 0][1]
            index_2 = compare.iloc[k, 1][1]
            # take the dot product of the corresponding review-level vectors to find the cosine similarity between them
            cos_sim.append((rev_embed_reviews[index_1] @ rev_embed_reviews[index_2]).item())
        # add cos_sim to 'compare' as a column, indicating the model and pooling function used
        compare[f'model{i}_pool{j}'] = cos_sim

## Display DataFrame.
display(compare)

,sentence_1,sentence_2,model0_pool0,model0_pool1,model1_pool0,model1_pool1,model2_pool0,model2_pool1
0,"(comfortable, 0)","(This is the most comfortable sofa ever., 1)",0.516279,0.477592,0.325693,0.421221,0.617525,0.661939
1,"(comfortable, 0)","(This is the most uncomfortable sofa ever., 2)",0.455039,0.463274,0.220350,0.360752,0.473663,0.543853
2,"(comfortable, 0)","(comfortable, 3)",1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
3,"(comfortable, 0)","(very comfortable, 4)",0.839744,0.808382,0.774429,0.802939,0.938370,0.975020
4,"(comfortable, 0)","(Loved this sofa, definitely a good one., 5)",0.352047,0.300140,0.333620,0.375301,0.627388,0.677291
5,"(comfortable, 0)","(Hated this sofa, comfortable but should not b...",0.385699,0.263491,0.113662,0.188081,0.436114,0.508968
6,"(comfortable, 0)","(Cozy and nice., 7)",0.657779,0.615678,0.568009,0.611809,0.852882,0.900629
7,"(comfortable, 0)","(Cozy but would not buy., 8)",0.514626,0.495506,0.173432,0.307923,0.611740,0.667207
8,"(This is the most comfortable sofa ever., 1)","(This is the most uncomfortable sofa ever., 2)",0.793414,0.820716,0.677423,0.744474,0.811101,0.836338
9,"(This is the most comfortable sofa ever., 1)","(comfortable, 3)",0.516279,0.477592,0.325693,0.421221,0.617525,0.661939


In the DataFrame 'compare', we see cosine similarity calculated across unique pairings of reviews, for each model/pooling combination. Higher cosine similarities indicate greater proximity in vector-space between the sentences, while lower cosine similarities indicate greater distance. Note that we have not pooled across the nine reviews; this chart of cosine similarities across pairs of reviews is meant to potentially offer insight into the combination of model/pooling that one might want to use.

For this project, I will select a model/pooling combination at random, using [Google's "Roll dice" application](https://share.google/DL6rzYIIXWeEjERj4). This process led me to choose model1 (nli-mpnet) and pool1 (CLS pooling).

### Scenario with Real-World Data

For our real-world data, I manually aggregated publicly-available customer reviews for three products from [Wayfair's website](https://www.wayfair.com/).

**Metadata:** The data was collected on Thursday, January 15, 2026. A search was made for 'comfortable sofa', and the results were sorted by 'Customer Rating'. It is worth noting that the results displayed were not replicable in a separate attempt using the same parameters. The reviews were sorted by 'Latest' and only ones with the 'Verified Buyer' tag were considered. I only included reviews in English (as far as I recognized). Some product information that seemed templated ("**Upholstery Color:** Gray" text, for example) and which was present in only some of the reviews was excluded. Under these conditions, the top 20 customer reviews were collected for each of three products.

The data will be loaded into a DataFrame with two columns: the associated product SKU, and the review.

In [2]:
wayfair = pd.read_csv('wayfair.csv')
# Make sure this file path is correct! 'wayfair.csv' is provided in the Github repo.
display(wayfair)

,sku,review_text
0,W008103065,This sofa is so beautiful and comfortable. I’m...
1,W008103065,This is a hit in our media room. It is comfort...
2,W008103065,I've had this couch for roughly one year and I...
3,W008103065,"Obsessed with this couch, I live on it, but wh..."
4,W008103065,Purchased quite a while back. Still in box as ...
5,W008103065,The sofa is very easy to assemble with clear i...
6,W008103065,My favourite couch I have ever owned. Love it-...
7,W008103065,Love this couch! Exactly what we were looking ...
8,W008103065,Perfect and very comfortable
9,W008103065,"Better than I expected, the back cushions are ..."


To begin, I will separate the reviews by product (identified by the SKU). If desired, another list of reviews, pertaining to another product, may be added here (see the TODO!).

In [3]:
# separating the reviews by product
W008 = wayfair.loc[wayfair.sku == 'W008103065', 'review_text'].to_list()
W011 = wayfair.loc[wayfair.sku == 'W011131433', 'review_text'].to_list()
W000 = wayfair.loc[wayfair.sku == 'W000902139', 'review_text'].to_list()

# a dictionary containing the products (TODO: can add another list of reviews to this dictionary as desired)
dat = {'W008': W008, 'W011': W011, 'W000': W000}

Now, I will apply the workflow described in 'The Hypothetical' to each product.

In [4]:
# loading our tokenizer/model and pooling choices
tokenizer = AutoTokenizer.from_pretrained(model_li[1])
model = AutoModel.from_pretrained(model_li[1])
pool_func = cls_pool

for item in dat:
    ## break each review into tokens
    dat[item] = tokenizer(dat[item], padding=True, truncation=True, return_tensors='pt')
    # creating a temp variable so that we can rewrite dat[item] with the output of our next step but hold onto the attention mask
    temp = dat[item]
    ## obtain a vectorized representation for each of these tokens
    with torch.no_grad():
        dat[item] = model(**dat[item])
    ## pool these representations to arrive at an aggregate representation for each review
    dat[item] = F.normalize(pool_func(dat[item].last_hidden_state, temp.attention_mask), p=2, dim=1)
    ## pool these review-level representations to arrive at an aggregate representations for the overall product
    # average across all of the reviews, unweighted average
    prod_pool = torch.sum(dat[item], 0) / dat[item].shape[0]
    dat[item] = F.normalize(prod_pool, p=2, dim=0)

And we've done it! We now have product-level representations for each of the three real-world sofas.

For fun, let's see how close (in terms of cosine similarity) that each sofa is to each of the others, as they are oriented in vector-space.

In [5]:
# pulls each product label from dat and pushes into list 'items'
items = []
for item in dat:
    items.append(item)

seen = []
for i in range(len(items)):
    for j in range(i + 1, len(items)):
        # checks if this is a unique pair of products
        if(not( ((i, j) in seen) or ((j, i) in seen)) ):
            # calculates the cosine similarity
            cos_sim = (dat[items[i]] @ dat[items[j]]).item()
            # prints the cosine similarity with labels
            print(f'SOFA {i} & SOFA {j}', round(cos_sim, 5))
            seen.append((i, j))

SOFA 0 & SOFA 1 0.96183
SOFA 0 & SOFA 2 0.95187
SOFA 1 & SOFA 2 0.94875


Just considering SOFAs 0-2, it appears that by cosine similarity, SOFAs 0 and 1 are the closest together in vector-space, and SOFAs 1 and 2 are the farthest apart. It's also worth noting that across all three sofas, the range of cosine similarities is relatively small. Perhaps this indicates that across the reviews that were collected, these sofas are described similarly?

### Discussion

I have been interested in this project for a while, extending what I learned in class about NLP workflows to embed not just text, but representations of products in vector-space and potentially gaining insight from this organization. My goal in this latest iteration of the project is to make clear my stance on some important issues that I think pertain to this work.

In previous versions of this project, I have talked a lot about the business, industry, and consumer applications of the demonstrated workflow, along with some issues. I think that it is important for me to be clear that I cannot endorse the use of this work in real-world systems, as there are a lot of issues that are inherent in this process that could have real effects for business owners and consumers, and perhaps others as well.

For example, different models understand language differently, and different people understand and use language differently. Systemic problems like discrimination by sex and by race are capable of manifesting in a system such as this - discrimination against certain ways of speaking, inequities surrounding who leaves comments and who doesn't. And there are other limitations as well - the weight of reviews relative to the quantity for a given product, the snowball effect of initial negative reviews, attention inequality driven by prominence or lack thereof in the rankings. All of these things, as well as other things that I have not brought up here, could have a real impact on sellers in an ecosystem like Wayfair or Amazon.

Additionally, I want to impress on readers that the representations that we are playing with here cannot necesarily be said to capture the product in its entirety. Perhaps natural language descriptions of a product offer a window of insight, but they do not necessarily capture the product in everything that it is. There is a mistake made, I think, in simplifying a product to solely these representations.

Finally, I wanted to talk about the notion of comparing products on the basis of a subjective attribute, which is something that I tried to demo in previous versions of this project. I removed that part of the project from this iteration, because I think that the way that some of the models seemed to work wasn't compatible with the way that I was trying to do that ranking. I also think that there is so much messiness in the idea of ranking on the basis of subjective attributes, inferred from natural language. I ended up switching to comparing where the products lie in vector-space relative to one another, which I think is an interesting insight to pull from this work.